# <span style="color:#e0bda8">1. Import Packages and Libraries</span>

In [1]:
# 1. Data Manipulation
# =====================================================
import pandas as pd
import numpy as np


# 2. Data Preprocessing
# =====================================================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline


# 3. Statistical Analysis
# =====================================================
from scipy.stats import skew, kurtosis


# 4. Path Configuration
# =====================================================
import os


# 5. Utilities
# =====================================================
import warnings

warnings.filterwarnings('ignore')



# <span style="color:#e0bda8">2. Project Structure and Directory Configuration </span>   

In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA = os.path.join(PROJECT_ROOT, "01_Data")
DATA_RAW = os.path.join(DATA, "01_Raw")
DATA_PROCESSED = os.path.join(DATA, "02_Processed")
DATA_STATS = os.path.join(DATA, "03_Statistics")
DATA_INDEX = os.path.join(DATA, "04_Index_Final")

CLUSTER_PCA = os.path.join(PROJECT_ROOT, "03_Clustering_PCA")
CLUSTER_PCA_EXCEL = os.path.join(CLUSTER_PCA, "01_Excel")
CLUSTER_PCA_FIG = os.path.join(CLUSTER_PCA, "02_Figures")

METHOD = os.path.join(PROJECT_ROOT, "04_Method_Comparison")
METHOD_EXCEL = os.path.join(METHOD, "01_Excel")
METHOD_FIG = os.path.join(METHOD, "02_Figures")

ML_RESULTS = os.path.join(PROJECT_ROOT, "05_ML_Results")
ML_METRICS = os.path.join(ML_RESULTS, "01_Metrics")
ML_PRED = os.path.join(ML_RESULTS, "02_Predictions")
ML_MODELS = os.path.join(ML_RESULTS, "03_Models")
ML_SHAP = os.path.join(ML_RESULTS, "04_SHAP")
ML_SHAP_EXCEL = os.path.join(ML_SHAP, "01_Excel")
ML_SHAP_FIG = os.path.join(ML_SHAP, "02_Figures")

# <span style="color:#e0bda8">3. Data Formatting </span>

In [3]:
df_raw = pd.read_csv(os.path.join(DATA_RAW, "esgdata.csv"))
framework = pd.read_csv(os.path.join(DATA_RAW, "framework.csv"))


In [4]:
env_framework = framework[framework['ESG Pillar'] == 'Environment'] 
env_indicators = list(env_framework['Indicator name'].unique())

soc_framework = framework[framework['ESG Pillar'] == 'Social']
soc_indicators = list(soc_framework['Indicator name'].unique())

gov_framework = framework[framework['ESG Pillar'] == 'Governance']
gov_indicators = list(gov_framework['Indicator name'].unique())

esg_indicators = env_indicators + soc_indicators + gov_indicators

In [5]:
cols_metadata = list(df_raw.loc[:, 'ISO3 code':'Indicator name'].columns)
cols_years = list(df_raw.loc[:, '2012':'2020'].columns)

df_framework = df_raw[df_raw['Indicator name'].isin(esg_indicators)]

df_selected = df_framework[cols_metadata + cols_years]

In [6]:
df_panel = df_selected.melt(id_vars=['ISO3 code', 'Economy', 'Indicator code', 'Indicator name'],
                  value_vars=[str(y) for y in range(2012, 2021)],
                  var_name='Year',
                  value_name='Value')

df_panel_wide = df_panel.pivot_table(
    index=['ISO3 code', 'Economy', 'Year'],   
    columns='Indicator name',               
    values='Value'
).reset_index()

df_panel_wide = df_panel_wide.set_index(['Economy', 'Year'])

In [7]:
europe_countries = ["Albania", "Armenia", "Austria", "Azerbaijan", "Belarus", "Belgium", "Bosnia and Herzegovina", 
                    "Bulgaria", "Croatia", "Cyprus", "Czechia", "Denmark", "Estonia", "Finland", "France", 
                    "Georgia", "Germany", "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Kazakhstan", 
                    "Latvia", "Lithuania", "Luxembourg", "Malta", "Moldova", "Montenegro", "Netherlands", 
                    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "Russian Federation", "Serbia", 
                    "Slovak Republic", "Slovenia", "Spain", "Sweden", "Switzerland", "Turkiye", "Ukraine", "United Kingdom"]

df_europe = df_panel_wide[df_panel_wide.index.get_level_values('Economy').isin(europe_countries)] 

In [8]:
vars_removed = ['ISO3 code', 'Primary Forest Loss', 'Children in employment, total (% of children ages 7-14)', 
                'Unmet need for contraception (% of married women ages 15-49)', 'Mammal species, threatened',
                'Literacy rate, adult total (% of people ages 15 and above)', 'Proportion of bodies of water with good ambient water quality',
                'Cause of death, by communicable diseases and maternal, prenatal and nutrition conditions (% of total)', 'Coastal protection'] 

vars_to_drop = [c for c in vars_removed if c in df_europe.columns]

df_clean = df_europe.drop(columns=vars_to_drop)
numeric_cols = df_clean.select_dtypes(include=np.number).columns


In [9]:
df_env_init = df_clean[[e for e in env_indicators if e in df_clean.columns]]
df_soc_init = df_clean[[s for s in soc_indicators if s in df_clean.columns]]
df_gov_init = df_clean[[g for g in gov_indicators if g in df_clean.columns]]

# <span style="color:#e0bda8">4. Pipeline Architecture </span>

## <span style="color:#e0bda8">4.1. Signal Aligning and Log-Transformation Decisions </span>

In [10]:
# Negative variables (36)

negative_vars = ['PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)',  
                'Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)', 
                'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)', 
                'Carbon dioxide (CO2) net fluxes from LULUCF - Total excluding non-tropical fires (Mt CO2e)',
                'Methane (CH4) emissions (total) excluding LULUCF (Mt CO2e)', 
                'Nitrous oxide (N2O) emissions (total) excluding LULUCF (Mt CO2e)',
                'Total greenhouse gas emissions excluding LULUCF (Mt CO2e)', 
                'Total greenhouse gas emissions per capita excluding LULUCF (t CO2e/capita)', 
                'Electricity production from coal sources (% of total)', 
                'Energy imports, net (% of energy use)', 
                'Energy intensity level of primary energy (MJ/$2017 PPP GDP)', 
                'Energy use (kg of oil equivalent per capita)', 
                'Fossil fuel energy consumption (% of total)', 
                'Cooling Degree Days', 
                'Heat Index 35', 
                'Heating Degree Days',
                'Land Surface Temperature', 
                'Level of water stress: freshwater withdrawal as a proportiondf of available freshwater resources', 
                'Population density (people per sq. km of land area)', 
                'Agricultural land (% of land area)', 
                'Agriculture, forestry, and fishing, value added (% of GDP)', 
                'Adjusted savings, natural resources depletion (% of GNI)', 
                'Adjusted savings, net forest depletion (% of GNI)', 
                'Annual freshwater withdrawals, total (% of internal resources)', 
                'Tree Cover Loss', 
                'Unemployment, total (% of total labor force) (modeled ILO estimate)', 
                'Mortality rate, under-5 (per 1,000 live births)', 
                'Prevalence of overweight (% of adults)', 
                'Prevalence of undernourishment (% of population)', 
                'Gini index', 
                'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)', 
                'Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)', 
                'Poverty headcount ratio at national poverty lines (% of population)', 
                'Prosperity gap (average shortfall from a prosperity standard of $28/day)'] 

# Centered variables (3)

centered_vars = ['Standardised Precipitation-Evapotranspiration Index', 
                'Fertility rate, total (births per woman)',  
                'School enrollment, primary and secondary (gross), gender parity index (GPI)'] 

# Positive variables (29)

positive_vars = ['Renewable electricity output (% of total electricity output)', 
                'Renewable energy consumption (% of total final energy consumption)', 
                'Forest area (% of land area)', 
                'Terrestrial and marine protected areas (% of total territorial area)', 
                'Food production index (2014-2016 = 100)',
                'Access to clean fuels and technologies for cooking  (% of population)', 
                'Access to electricity (% of population)', 
                'People using safely managed drinking water services (% of population)', 
                'People using safely managed sanitation services (% of population)', 
                'Life expectancy at birth, total (years)', 
                'Population ages 65 and above (% of total population)',
                'Government expenditure on education, total (% of government expenditure)', 
                'School enrollment, primary (% gross)',
                'Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)',
                'Hospital beds (per 1,000 people)',
                'Income share held by lowest 20%', 
                'GDP (annual % growth)', 
                'Individuals using the Internet (% of population)',
                'Proportion of seats held by women in national parliaments (%)', 
                'Ratio of female to male labor force participation rate (%) (modeled ILO estimate)',
                'Government Effectiveness: Estimate', 
                'Regulatory Quality: Estimate', 
                'Economic and Social Rights Performance Score', 
                'Voice and Accountability: Estimate',
                'Patent applications, residents', 
                'Research and development expenditure (% of GDP)', 
                'Scientific and technical journal articles',
                'Control of Corruption: Estimate', 
                'Net migration', 
                'Political Stability and Absence of Violence/Terrorism: Estimate', 
                'Rule of Law: Estimate'] 


transformation_vars= [
    "Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)",
    "Carbon dioxide (CO2) net fluxes from LULUCF - Total excluding non-tropical fires (Mt CO2e)",  
    "Methane (CH4) emissions (total) excluding LULUCF (Mt CO2e)",
    "Nitrous oxide (N2O) emissions (total) excluding LULUCF (Mt CO2e)",
    "Total greenhouse gas emissions excluding LULUCF (Mt CO2e)",
    "Tree Cover Loss",
    "Energy use (kg of oil equivalent per capita)",
    "Energy imports, net (% of energy use)",  
    "PM2.5 air pollution, mean annual exposure (micrograms per cubic meter)", 
    "Cooling Degree Days",  
    "Heating Degree Days",  
    "Mortality rate, under-5 (per 1,000 live births)",  
    "Fertility rate, total (births per woman)", 
    "Prevalence of undernourishment (% of population)",  
    "Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)", 
    "Poverty headcount ratio at $8.30 a day (2021 PPP) (% of population)",  
    "Prosperity gap (average shortfall from a prosperity standard of $28/day)",
    "Patent applications, residents",
    "Scientific and technical journal articles",
    "Population density (people per sq. km of land area)",
    "Adjusted savings, natural resources depletion (% of GNI)",  
    "Adjusted savings, net forest depletion (% of GNI)", 
    "Annual freshwater withdrawals, total (% of internal resources)"  
]
                
ref_values = {
    'Standardised Precipitation-Evapotranspiration Index': 0,
    'Fertility rate, total (births per woman)': 2.1,
    'School enrollment, primary and secondary (gross), gender parity index (GPI)': 1.0
}

## <span style="color:#e0bda8">4.2. Classes Creation </span>

In [11]:
from sklearn.base import BaseEstimator, TransformerMixin

class TemporalInterpolationTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, group_col='Economy', time_col='Year', numeric_cols=None):
        self.group_col = group_col
        self.time_col = time_col
        self.numeric_cols = numeric_cols

    def fit(self, X, y=None):
        if self.numeric_cols is None:
            self.numeric_cols_ = X.select_dtypes(include='number').columns.tolist()
        else:
            self.numeric_cols_ = self.numeric_cols
        return self

    def transform(self, X):
        X = X.copy().reset_index()
        X['Year'] = pd.to_datetime(X['Year'], format='%Y')
                
        def interpolate_group(group):
            group[self.group_col] = group.name
            original_index = group.index
            group = group.set_index(self.time_col)
            group[self.numeric_cols_] = (
                group[self.numeric_cols_]
                .interpolate(method='time')
                .ffill()
                .bfill()
            )

            group = group.reset_index()
            group.index = original_index
            return group

        X = X.groupby(self.group_col, group_keys=False).apply(interpolate_group, include_groups=False)
        X['Year'] = X['Year'].dt.year.astype(int)
        X = X.set_index(['Economy','Year'])
        
        return(X)

class KNNImputerSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_neighbors=5, weights='distance'):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.imputer = KNNImputer(n_neighbors=self.n_neighbors, weights=self.weights)
    
    def fit(self, X, y=None):
        self.imputer.fit(X)
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        mask_missing = X_copy.isna()
        
        # Imputar
        X_imputed_array = self.imputer.transform(X_copy)
        
        # Converter de volta para DataFrame com mesmo índice e colunas
        X_imputed = pd.DataFrame(X_imputed_array, index=X_copy.index, columns=X_copy.columns)
        
        # Substituir apenas os missing
        X_copy[mask_missing] = X_imputed[mask_missing]
        return X_copy
    
class SafeStandardScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None):
        self.columns = columns
        self.scaler = StandardScaler()
    
    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns])
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.columns] = self.scaler.transform(X_copy[self.columns])
        return X_copy
    
class SafeMinMaxScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, feature_range=(0, 100)):
        self.columns = columns
        self.scaler = MinMaxScaler(feature_range=feature_range)
    
    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns])
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        X_copy[self.columns] = self.scaler.transform(X_copy[self.columns])
        return X_copy
    
class SignedLog1pTransformer(BaseEstimator, TransformerMixin):
    """
    Transformer que aplica a Signed Log1p Transformation:
    f(x) = sign(x) * log1p(abs(x))
    Funciona para valores positivos e negativos.
    """
    def __init__(self, vars_to_transform=None):
        self.vars_to_transform = vars_to_transform
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        
        for var in self.vars_to_transform or []:
            if var not in X_copy.columns:
                continue
            
            # Aplicar signed log1p
            X_copy[var] = np.sign(X_copy[var]) * np.log1p(np.abs(X_copy[var]))
        
        return X_copy
    
class SignalAligningTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, negative_vars=None, centered_vars=None, ref_values=None):
        self.negative_vars = negative_vars
        self.centered_vars = centered_vars
        self.ref_values = ref_values
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()

        existing_negative_vars = [v for v in self.negative_vars if v in X_copy.columns]
        if existing_negative_vars:
            X_copy[existing_negative_vars] = -1 * X_copy[existing_negative_vars]

        if self.centered_vars and self.ref_values:
            for var in self.centered_vars:
                if var in X_copy.columns:
                    X_copy[var] = np.abs(X_copy[var] - self.ref_values[var])
                    X_copy[var] = -1 * X_copy[var]
        return X_copy

# <span style="color:#e0bda8">5. Pre-Processing </span>

In [12]:
env_pipeline_pca = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeStandardScaler(columns=df_env_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

soc_pipeline_pca = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeStandardScaler(columns=df_soc_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

gov_pipeline_pca = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeStandardScaler(columns=df_gov_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

env_pipeline_score = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeMinMaxScaler(columns=df_env_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

soc_pipeline_score = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars,
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeMinMaxScaler(columns=df_soc_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

gov_pipeline_score = Pipeline(steps=[
    ('temporal_interpolation', TemporalInterpolationTransformer(group_col='Economy')), 
    ("log1p", SignedLog1pTransformer(vars_to_transform=transformation_vars)),
    ('signal_aligning', SignalAligningTransformer(
        negative_vars=negative_vars, 
        centered_vars=centered_vars,
        ref_values=ref_values)),
    ('scaler', SafeMinMaxScaler(columns=df_gov_init.columns)),
    ('knn_imputer', KNNImputerSafe(n_neighbors=5, weights='distance'))
])

# Apply pipelines to the cleaned dataframes
env_pca = env_pipeline_pca.fit_transform(df_env_init)
df_env_pca = pd.DataFrame(
    env_pca,
    index=env_pca.index,
    columns=env_pca.columns
)

env_score = env_pipeline_score.fit_transform(df_env_init)
df_env_score = pd.DataFrame(
    env_score,
    index=env_score.index,
    columns=env_score.columns
)

soc_pca = soc_pipeline_pca.fit_transform(df_soc_init)
df_soc_pca = pd.DataFrame(
    soc_pca,
    index=soc_pca.index,
    columns=soc_pca.columns
)

soc_score = soc_pipeline_score.fit_transform(df_soc_init)
df_soc_score = pd.DataFrame(
    soc_score,
    index=soc_score.index,
    columns=soc_score.columns
)

gov_pca = gov_pipeline_pca.fit_transform(df_gov_init)
df_gov_pca = pd.DataFrame(
    gov_pca,
    index=gov_pca.index,
    columns=gov_pca.columns
)

gov_score = gov_pipeline_score.fit_transform(df_gov_init)
df_gov_score = pd.DataFrame(
    gov_score,
    index=gov_score.index,
    columns=gov_score.columns
)



In [13]:
df_env_init.to_csv(os.path.join(DATA_PROCESSED, "df_env_init.csv"))
df_soc_init.to_csv(os.path.join(DATA_PROCESSED, "df_soc_init.csv"))
df_gov_init.to_csv(os.path.join(DATA_PROCESSED, "df_gov_init.csv"))

df_env_pca.to_csv(os.path.join(DATA_PROCESSED, 'df_env_pca.csv'))
df_soc_pca.to_csv(os.path.join(DATA_PROCESSED, 'df_soc_pca.csv'))
df_gov_pca.to_csv(os.path.join(DATA_PROCESSED, 'df_gov_pca.csv'))

df_env_score.to_csv(os.path.join(DATA_PROCESSED, 'df_env_score.csv'))
df_soc_score.to_csv(os.path.join(DATA_PROCESSED, 'df_soc_score.csv'))
df_gov_score.to_csv(os.path.join(DATA_PROCESSED, 'df_gov_score.csv'))

# <span style="color:#e0bda8">6. Restoring the Dataset to its Original Scale </span>

### <span style="color:#e0bda8">6.0. Auxiliary Functions </span>

In [14]:
def signed_log1p_inverse(X, vars_to_transform):
    X_copy = X.copy()
    for var in vars_to_transform or []:
        if var not in X_copy.columns:
            continue
        X_copy[var] = np.sign(X_copy[var]) * (np.expm1(np.abs(X_copy[var])))
    return X_copy

def signal_aligning_inverse(X, negative_vars=None, centered_vars=None, ref_values=None):
    X_copy = X.copy()
    
    # Inverter variáveis "negativas"
    existing_negative_vars = [v for v in negative_vars or [] if v in X_copy.columns]
    if existing_negative_vars:
        X_copy[existing_negative_vars] = -1 * X_copy[existing_negative_vars]
    
    # Inverter variáveis centralizadas
    if centered_vars and ref_values:
        for var in centered_vars:
            if var in X_copy.columns:
                X_copy[var] = ref_values[var] + (-1 * X_copy[var])
                
    return X_copy

def inverse_pipeline(df_transformed, pipeline, vars_to_transform=None, negative_vars=None, centered_vars=None, ref_values=None):
    X_inv = df_transformed.copy()
    
    # 1. Inverter scaler
    scaler_step = pipeline.named_steps['scaler']
    X_inv[scaler_step.columns] = scaler_step.scaler.inverse_transform(X_inv[scaler_step.columns])
    
    # 2. Inverter SignalAligning
    X_inv = signal_aligning_inverse(X_inv, negative_vars=negative_vars, centered_vars=centered_vars, ref_values=ref_values)
    
    # 3. Inverter SignedLog1p
    X_inv = signed_log1p_inverse(X_inv, vars_to_transform)
    
    return X_inv

## <span style="color:#e0bda8">6.1. Final Dataframes on the Original Scale </span>

In [15]:
df_env_unscaled = inverse_pipeline(
    df_env_score, 
    env_pipeline_score, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

df_soc_unscaled = inverse_pipeline(
    df_soc_score, 
    soc_pipeline_score, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

df_gov_unscaled = inverse_pipeline(
    df_gov_score, 
    gov_pipeline_score, 
    vars_to_transform=transformation_vars, 
    negative_vars=negative_vars,
    centered_vars=centered_vars, 
    ref_values=ref_values
)

# <span style="color:#e0bda8">7. Summary Statistics </span>

In [16]:
def full_summary(df):
    summary_list = []
    
    for col in df.columns:
        dtype = df[col].dtype
        
        if pd.api.types.is_numeric_dtype(df[col]):
            col_mean = df[col].mean()
            col_std = df[col].std()
            col_min = df[col].min()
            col_25 = df[col].quantile(0.25)
            col_50 = df[col].quantile(0.5)
            col_75 = df[col].quantile(0.75)
            col_max = df[col].max()
            col_skew = skew(df[col].dropna())
            col_kurt = kurtosis(df[col].dropna())
        else:
            col_mean = col_std = col_min = col_25 = col_50 = col_75 = col_max = col_skew = col_kurt = None
        
        summary_list.append({
            'Variable': col,
            'Data type': dtype,
            'Mean': col_mean,
            'Std': col_std,
            'Min': col_min,
            '25%': col_25,
            '50%': col_50,
            '75%': col_75,
            'Max': col_max,
            'Skewness': col_skew,
            'Kurtosis': col_kurt
        })
    
    summary_df = pd.DataFrame(summary_list)
    return summary_df.round(2)



In [17]:
summary_env = full_summary(df_env_unscaled)
summary_soc = full_summary(df_soc_unscaled)
summary_gov = full_summary(df_gov_unscaled)

summary =  pd.concat(
    [summary_env, summary_soc, summary_gov],
    keys=['Environment', 'Social', 'Governance']
)

In [18]:
pd.set_option('display.max_rows', None)

summary

Variable Data type  \
Environment 0   PM2.5 air pollution, mean annual exposure (mic...   float64   
            1   Carbon dioxide (CO2) emissions (total) excludi...   float64   
            2   Carbon dioxide (CO2) emissions excluding LULUC...   float64   
            3   Carbon dioxide (CO2) net fluxes from LULUCF - ...   float64   
            4   Methane (CH4) emissions (total) excluding LULU...   float64   
            5   Nitrous oxide (N2O) emissions (total) excludin...   float64   
            6   Total greenhouse gas emissions excluding LULUC...   float64   
            7   Total greenhouse gas emissions per capita excl...   float64   
            8   Electricity production from coal sources (% of...   float64   
            9               Energy imports, net (% of energy use)   float64   
            10  Energy intensity level of primary energy (MJ/$...   float64   
            11       Energy use (kg of oil equivalent per capita)   float64   
            12        Fossil fuel energy consumption (% of total)   float64   
            13  Renewable electricity output (% of total elect...   float64   
            14  Renewable energy consumption (% of total final...   float64   
            15                                Cooling Degree Days   float64   
            16                                      Heat Index 35   float64   
            17                                Heating Degree Days   float64   
            18                           Land Surface Temperature   float64   
            19  Level of water stress: freshwater withdrawal a...   float64   
            20  Population density (people per sq. km of land ...   float64   
            21  Standardised Precipitation-Evapotranspiration ...   float64   
            22                 Agricultural land (% of land area)   float64   
            23  Agriculture, forestry, and fishing, value adde...   float64   
            24            Food production index (2014-2016 = 100)   float64   
            25  Adjusted savings, natural resources depletion ...   float64   
            26  Adjusted savings, net forest depletion (% of GNI)   float64   
            27  Annual freshwater withdrawals, total (% of int...   float64   
            28                       Forest area (% of land area)   float64   
            29  Terrestrial and marine protected areas (% of t...   float64   
            30                                    Tree Cover Loss   float64   
Social      0   Access to clean fuels and technologies for coo...   float64   
            1             Access to electricity (% of population)   float64   
            2   People using safely managed drinking water ser...   float64   
            3   People using safely managed sanitation service...   float64   
            4            Fertility rate, total (births per woman)   float64   
            5             Life expectancy at birth, total (years)   float64   
            6   Population ages 65 and above (% of total popul...   float64   
            7   Government expenditure on education, total (% ...   float64   
            8                School enrollment, primary (% gross)   float64   
            9   Labor force participation rate, total (% of to...   float64   
            10  Unemployment, total (% of total labor force) (...   float64   
            11                   Hospital beds (per 1,000 people)   float64   
            12    Mortality rate, under-5 (per 1,000 live births)   float64   
            13             Prevalence of overweight (% of adults)   float64   
            14   Prevalence of undernourishment (% of population)   float64   
            15                                         Gini index   float64   
            16                    Income share held by lowest 20%   float64   
            17  Poverty headcount ratio at $3.00 a day (2021 P...   float64   
            18  Poverty headcount ratio at $8.30 a day (2021 P...   float64   
            19  Poverty head

In [19]:
summary.to_excel(os.path.join(DATA_STATS, "Summary_Statistics.xlsx"))